https://github.com/vlawhern/arl-eegmodels

[EEGNet](https://github.com/aliasvishnu/EEGNet)

In [1]:
from BCI_ToolBox.models.EEGNet import EEGNet
from BCI_ToolBox.models.DeepConvNet import DeepConvNet


/home/saul_park/miniconda3/envs/EEGToText/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### EEGNet

In [3]:
import torch
import torch.nn as nn

# BCI_ToolBox 라이브러리의 클래스들을 import 했다고 가정
# from BCI_ToolBox.base.layers import Conv2dWithConstraint, LinearWithConstraint
# from BCI_ToolBox.utils.utils import initialize_weight
# class EEGNet(nn.Module): ...

# 1. 데이터 차원 정의
depth = 512
sequence_length = 56

# 2. 모델의 입력 형태(input_shape) 정의
# 형식: (batch_size, channels, height, width)
# 여기서는 (None, 1, depth, sequence_length)에 해당합니다.
# batch_size는 보통 None이나 예시 숫자를 넣습니다.
input_shape = (1, 1, depth, sequence_length)

# 3. 모델 생성
# n_classes는 최종적으로 분류할 클래스의 개수입니다. (예: 2개)
n_classes = 2

# EEGNet 모델 객체를 생성합니다.
model = EEGNet(
    n_classes=n_classes,
    input_shape=input_shape,
    F1=16,                # 첫 번째 Temporal Conv 필터 수
    D=2,                  # Spatial Conv 깊이 승수
    F2='auto',            # F1 * D = 32
    T1=32,                # 첫 번째 Temporal Conv 커널 크기 (sequence_length보다 작아야 함)
    T2=16,                # 두 번째 Temporal Conv 커널 크기
    P1=4,                 # 첫 번째 Pooling 크기
    P2=8,                 # 두 번째 Pooling 크기
    drop_out=0.5,         # 드롭아웃 비율
    pool_mode='mean',     # 풀링 방식
    weight_init_method='xavier_uniform'
)

# 더미 데이터로 테스트
dummy_input = torch.randn(16, sequence_length, depth) # (batch, channel, height, width)
dummy_input = dummy_input.permute(0, 2, 1).unsqueeze(1)  # (batch, channel, height, width)
output = model(dummy_input)
print(f"\nInput shape: {dummy_input.shape}")
print(f"Output shape: {output.shape}")


Input shape: torch.Size([16, 1, 512, 56])
Output shape: torch.Size([16, 56, 1024])


/home/saul_park/miniconda3/envs/EEGToText/lib/python3.9/site-packages/torch/nn/modules/conv.py:439: UserWarning: Using padding='same' with even kernel lengths and odd dilation may require a zero-padded copy of the input be created (Triggered internally at  /opt/conda/conda-bld/pytorch_1623448255797/work/aten/src/ATen/native/Convolution.cpp:660.)
  return F.conv2d(input, weight, bias, self.stride,


### DeepConvNet

In [2]:
import torch
import torch.nn as nn

# --- 논문(Figure 1) 기반 파라미터 설정 ---
# 🧠 파라미터 설명:
# [cite_start]n_classes: 4개의 클래스(왼손, 오른손, 발, 휴식) [cite: 119]
# input_shape: [배치, 채널, 전극 수, 시간 샘플]. [cite_start]Figure 1의 입력 크기(44 electrodes)를 따름 [cite: 191]
# [cite_start]first_conv_length: 첫 번째 시간적 합성곱 커널 크기 [cite: 222]
# [cite_start]block_out_channels: 각 합성곱 블록의 출력 채널 수 [cite: 192, 203, 207, 211]
# [cite_start]pool_size: 맥스 풀링의 커널 및 스트라이드 크기 [cite: 200, 205, 209, 212]
# last_dim: 마지막 블록의 출력을 Flatten한 후의 크기. [cite_start]Figure 1의 마지막 특징 맵(200x1x6)을 기반으로 계산 [cite: 238, 239, 241]
params = {
    'input_shape': [32, 1, 512, 56], # [Batch, In-channels, Electrodes, Samples]
    'first_conv_length': 5,
    'block_out_channels': [32, 64, 128, 256, 512],
    'pool_size': 3,
}

# --- 모델 불러오기 ---
print("논문 기반 파라미터로 DeepConvNet 모델을 생성합니다.")
print(f"사용된 파라미터: {params}")
model = DeepConvNet(**params)
print("\n모델 생성 완료!")
# print(model)

# --- 테스트 입력으로 모델 실행 ---
# input_shape과 동일한 크기의 더미 텐서 생성
dummy_input = torch.randn(params['input_shape'])
print(f"\n더미 입력 (크기: {dummy_input.shape})으로 모델을 테스트합니다...")
output = model(dummy_input)
print(f"모델 출력 크기: {output.shape}") # 예상 출력 크기: [1, 4]

논문 기반 파라미터로 DeepConvNet 모델을 생성합니다.
사용된 파라미터: {'input_shape': [32, 1, 512, 56], 'first_conv_length': 5, 'block_out_channels': [32, 64, 128, 256, 512], 'pool_size': 3}

모델 생성 완료!

더미 입력 (크기: torch.Size([32, 1, 512, 56]))으로 모델을 테스트합니다...
모델 출력 크기: torch.Size([32, 512, 56])


/home/saul_park/miniconda3/envs/EEGToText/lib/python3.9/site-packages/torch/nn/functional.py:718: UserWarning: Named tensors and all their associated APIs are an experimental feature and subject to change. Please do not use them for anything important until they are released as stable. (Triggered internally at  /opt/conda/conda-bld/pytorch_1623448255797/work/c10/core/TensorImpl.h:1156.)
  return torch.max_pool2d(input, kernel_size, stride, padding, dilation, ceil_mode)
